# 07 — Case Classes & Enums

Notebook 06 gave you classes, objects, and traits — the imperative-OOP shape. This notebook gives you the form most real Scala code reaches for: **case classes** for immutable data records, and **enums** for finite sets of alternatives. Together they are Scala's expression of *algebraic data types* — the backbone of domain modelling in the language.

Two ideas to anchor on:

- A **case class** is a *product* type — a record bundling several fields together. *And this AND this AND this.*
- An **enum** is a *sum* type — a choice between a fixed set of alternatives. *Either this OR this OR this.*

Most real-world domains are built by combining the two. Pattern matching in notebook 08 is how you take them apart again.

## `case class` — immutable records, batteries included

Prefix `class` with the `case` keyword and Scala 3 does a remarkable amount of work for you.

In [ ]:
case class User(name: String, email: String, age: Int)

val u = User("ganesh", "g@example.com", 35)
u.name        // "ganesh"
u.email       // "g@example.com"
u.age         // 35

That one line gives you, for free:

1. **Public `val` fields** for every constructor parameter — no `val` prefix required.
2. A companion object with `apply`, so `User(...)` works without `new`.
3. A companion `unapply` method that powers pattern matching (notebook 08).
4. A sensible `toString` — `User(ganesh, g@example.com, 35)`, not `User@7a3b1c`.
5. **Structural `equals` and `hashCode`** — two instances with the same field values are equal.
6. A `copy` method for creating a modified version without mutating the original.
7. Automatic serialization (the compiler marks case classes `Serializable`) — useful in Spark.

That list is the difference between a `case class` and a regular `class`. You almost never write a plain `class` for data; you write a `case class`.

## `copy` — derive a modified instance

Case classes are immutable. To get a *changed* version, you call `copy`, which uses named parameters with defaults so you only mention the fields you want to change.

In [ ]:
val u = User("ganesh", "g@example.com", 35)

val older = u.copy(age = 36)
// older: User = User(ganesh, g@example.com, 36)

val renamed = u.copy(name = "Ganesh M", email = "new@example.com")
// renamed: User = User(Ganesh M, new@example.com, 35)

u                // still User(ganesh, g@example.com, 35) — unchanged
u eq older       // false — different instances

`copy` is the immutable update pattern. Think of it as *the same record, with these fields different*. It is how you express state change in idiomatic Scala — produce a new value, don't mutate the old one. Notebook 09 will lean on `copy` again when modelling error-handling flows.

## Structural equality

Plain classes use **reference** equality — two instances are equal only if they point to the same object in memory. Case classes use **structural** equality — two instances are equal if all their fields are equal.

In [ ]:
case class Point(x: Int, y: Int)

val a = Point(1, 2)
val b = Point(1, 2)
val c = a

a == b           // true — same field values
a eq b           // false — different objects in memory
a eq c           // true — same object reference

Two things to register:

- **`==` is structural equality.** It calls the `equals` method, which case classes override to compare fields. This is the opposite of Java, where `==` on objects is reference equality.
- **`eq` is reference equality.** Rarely needed. Reach for it only when you specifically care whether two names point to the same object.

Structural equality also gives you a working `hashCode`, which means case classes Just Work as keys in `Map`, elements in `Set`, and identifiers across distributed systems.

## Defaults, named arguments, multi-field

Case class parameters can have default values, and callers can pass arguments by name. The same rules as method parameters from notebook 03.

In [ ]:
case class Server(host: String, port: Int = 8080, useTls: Boolean = false)

Server("localhost")
// Server(localhost, 8080, false)

Server("prod.example.com", port = 443, useTls = true)
// Server(prod.example.com, 443, true)

Defaults turn a case class into a small, self-documenting configuration record. Named arguments at the call site read like prose: `Server("host", port = 443)` is unambiguous in a way a positional `Server("host", 443)` is not.

## `case object` — the zero-field case class

When a case has no fields — a singleton that participates in pattern matching and structural equality the same way case classes do — use a **`case object`**.

In [ ]:
case object Empty

Empty.toString       // "Empty"
Empty == Empty       // true

You'll see `case object` most often as one of the alternatives of a sum type — the cases that carry no data. With Scala 3's `enum`, you usually don't have to write `case object` by hand anymore; the enum form handles both shapes uniformly. We'll see this in a moment.

## `enum` — a finite set of named alternatives

Scala 3 introduced a proper `enum` keyword. The simplest use is a closed set of named values — what Java's `enum` does.

In [ ]:
enum Color:
  case Red, Green, Blue

val c: Color = Color.Green
c                            // Green
c.ordinal                    // 1
Color.values                 // Array(Red, Green, Blue)
Color.valueOf("Red")         // Red

Three useful members come along for free:

- `ordinal` — the zero-based position of a case.
- `Color.values` — an array of every case, in declaration order.
- `Color.valueOf("Red")` — look up by name; throws if the name isn't a case.

But Scala 3 enums go further than Java's. Each case can also carry **its own fields and types**.

## Parameterised enum cases — the real ADT

When cases carry data, the enum becomes a true *sum type*: a value is one of several alternatives, and each alternative has its own shape.

In [ ]:
enum Shape:
  case Circle(radius: Double)
  case Rectangle(width: Double, height: Double)
  case Triangle(base: Double, height: Double)

val s1: Shape = Shape.Circle(2.0)
val s2: Shape = Shape.Rectangle(3.0, 4.0)

Each case is essentially a case class. `Shape.Circle(2.0)` is a value of type `Shape`, specifically of subtype `Shape.Circle`, with one field `radius`. It has structural equality, a sensible `toString`, a `copy` method, and pattern matching support — the same package as a regular case class.

What you've just declared is the canonical algebraic data type:

```
  Shape = Circle(radius)
        | Rectangle(width, height)
        | Triangle(base, height)
```

*A shape is a Circle OR a Rectangle OR a Triangle.* And each option carries its own data. This compositional shape is what notebook 08's pattern matching exists to consume.

## Mixing data and no-data cases

An enum can freely mix cases that carry data and cases that don't. The classic example is an `Option`-shaped type.

In [ ]:
enum Maybe[+A]:
  case Some(value: A)
  case None

val a: Maybe[Int] = Maybe.Some(42)
val b: Maybe[Int] = Maybe.None

`Some` carries a value; `None` does not. Scala compiles the no-data case to a singleton (the same as a `case object`) and the data case to a case-class-like subtype. The whole enum represents *zero-or-one value of type A*.

That `+A` is a **variance annotation** — we'll explain it properly in notebook 10. For now, read it as *Maybe is covariant in A*, which lets `Maybe[Int]` be used where `Maybe[Any]` is expected.

The standard library's `Option[A]`, which you've been meeting since notebook 04 (`Map.get`), is structurally exactly this shape.

## Enums can have methods too

An enum can declare methods that all cases inherit, just like a trait. Cases can also override them.

In [ ]:
enum Shape:
  case Circle(radius: Double)
  case Rectangle(width: Double, height: Double)

  def area: Double = this match
    case Circle(r)       => math.Pi * r * r
    case Rectangle(w, h) => w * h

Shape.Circle(2.0).area           // 12.566...
Shape.Rectangle(3.0, 4.0).area   // 12.0

Two things appearing for the first time:

- **`this match`** — the start of a pattern match against the current value. Notebook 08 unpacks this fully.
- **Methods on the enum itself.** They live on every case. The `area` method uses pattern matching to decide which formula to apply.

This is how you put behaviour next to the data shape it operates on, while keeping the data shape closed (no surprise new cases can be added by a downstream library).

## Product + Sum = ADTs

Most real domain types are products inside sums. A trading event might be:

```
  Event = Trade(symbol, price, quantity)        <- product
        | Quote(symbol, bid, ask)               <- product
        | Heartbeat(timestamp)                  <- product
                ^                ^
                |                |
             sum (the |s)     product (the fields inside each case)
```

You express that in Scala as:

In [ ]:
enum Event:
  case Trade(symbol: String, price: Double, quantity: Int)
  case Quote(symbol: String, bid: Double, ask: Double)
  case Heartbeat(timestamp: Long)

Three things you have automatically by writing those four lines:

1. The compiler knows the **full list of alternatives**. Pattern matches that miss a case will trigger an exhaustiveness warning.
2. Each alternative has named fields, equality, copy, and a clean `toString`.
3. Adding a new alternative — say `Cancel(orderId: Long)` — is a single line, and the compiler tells you every place that needs to handle it.

That last point is huge. It turns the type system into a worklist. Add a case; the compiler enumerates the broken sites. This is what *type-safe domain modelling* means in practice.

## When to reach for what

| You want | Pick |
|---|---|
| A record with several fields, no alternatives | `case class` |
| A finite set of named tags, no data | `enum` with only no-arg cases |
| A choice between several shapes, each carrying its own data | `enum` with parameterised cases |
| Behaviour mixable into many unrelated types | `trait` (notebook 06) |
| Per-instance mutable state | `class` (notebook 06) |

## Putting it together

A small but realistic model: a payment system that knows three payment methods and produces an outcome that is either a success or one of two failure modes.

In [ ]:
enum PaymentMethod:
  case Card(number: String, expiry: String)
  case BankTransfer(iban: String)
  case Wallet(provider: String, accountId: String)

enum PaymentResult:
  case Success(transactionId: String, amount: Double)
  case Declined(reason: String)
  case Failed(error: String)

case class Payment(
  amount: Double,
  method: PaymentMethod,
  result: PaymentResult,
)

val p = Payment(
  amount = 49.99,
  method = PaymentMethod.Card("4111...", "12/27"),
  result = PaymentResult.Success("txn-001", 49.99),
)

// derive a refunded version without mutating p
val refunded = p.copy(result = PaymentResult.Failed("refunded"))

Notice the composition:

- `Payment` is a **product** — one record with three fields.
- `PaymentMethod` and `PaymentResult` are **sums** — each is one of several alternatives.
- One of `Payment`'s fields *is* a sum (`PaymentResult`). Products containing sums containing products is the normal shape of domain data.
- `copy` lets us derive `refunded` from `p` without touching the original.

Walking `p.result` to decide what to do with it — log the success, alert on the failure, retry the decline — is exactly the job pattern matching was designed for.

## What's next

Notebook 08 introduces **pattern matching** — the way you take case classes and enums apart. It is where the compiler-enforced exhaustiveness kicks in: write a `match` against `PaymentResult`, miss the `Declined` case, and the compiler tells you. The `match` keyword is the partner to every `enum` you wrote in this notebook.